## Applying ToMe to timm models
A quick example showing how to apply ToMe to a timm model off-the-shelf.

In [ ]:
!git clone https://github.com/facebookresearch/ToMe.git
!pwd

In [ ]:
%cd /content/ToMe

In [ ]:
!pip install timm

In [ ]:
import timm
import tome
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
import torch

from PIL import Image

In [ ]:
# Use any ViT model here (see timm.models.vision_transformer)
model_name = "vit_base_patch16_224"

# Load a pretrained model
model = timm.create_model(model_name, pretrained=True)

In [ ]:
input_size = model.default_cfg["input_size"][1]

transform = transforms.Compose([
    transforms.Resize(int((256 / 224) * input_size), interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(input_size),
    transforms.ToTensor(),
    transforms.Normalize(model.default_cfg["mean"], model.default_cfg["std"]),
])

In [ ]:
img = Image.open("examples/images/husky.png")
img_tensor = transform(img)[None, ...]

img  # Note: image generated with stable diffusion

In [ ]:
# Run the image through the baseline model
# 248: Eskimo dog, husky
# 249: malamute, malemute, Alaskan malamute
# 250: Siberian husky
model(img_tensor).topk(5).indices[0].tolist()

### Applying ToMe
Simply patch the model after initialization to enable ToMe.

In [ ]:
# Patch the model with ToMe
tome.patch.timm(model)

In [ ]:
# Run the model with no reduction (should be the same as before)
model.r = 0
model(img_tensor).topk(5).indices[0].tolist()

In [ ]:
# Run the model with some reduction
# Only the least applicable class changed
model.r = 8
model(img_tensor).topk(5).indices[0].tolist()

In [ ]:
# Run the model with a lot of reduction
# Top-3 most applicable classes didn't change (husky, Siberian husky, Alaskan malamute)
# But model is 2x faster now! See 1_benchmark_timm.ipynb
model.r = 16
model(img_tensor).topk(5).indices[0].tolist()

## Benchmarking ToMe
We provide some benchmarking code in order to benchmark ToMe's throughput.

**Note**: notebooks have an overhead. To properly benchmark, use a standalone script.

In [ ]:
# Use any ViT model here (see timm.models.vision_transformer)
model_name = "vit_base_patch16_224"

# Load a pretrained model
model = timm.create_model(model_name, pretrained=True)

In [ ]:
# Set this to be whatever device you want to benchmark on
# If you don't have a GPU, you can use "cpu" but you probably want to set the # runs to be lower

if torch.cuda.is_available():
  device = torch.device("cuda")
  print('Training on GPU')
else:
  device = torch.device("cpu")
  print('Training on CPU')

runs = 50
batch_size = 256  # Lower this if you don't have that much memory
input_size = model.default_cfg["input_size"]

In [ ]:
# Baseline benchmark
baseline_throughput = tome.utils.benchmark(
    model,
    device=device,
    verbose=True,
    runs=runs,
    batch_size=batch_size,
    input_size=input_size
)

### Applying ToMe
Simply patch the model after initialization to enable ToMe.

In [ ]:
# Apply ToMe
tome.patch.timm(model)

In [ ]:
# ToMe with r=16
model.r = 16
tome_throughput = tome.utils.benchmark(
    model,
    device=device,
    verbose=True,
    runs=runs,
    batch_size=batch_size,
    input_size=input_size
)
print(f"Throughput improvement: {tome_throughput / baseline_throughput:.2f}x")

In [ ]:
# ToMe with r=16 and a decreasing schedule
model.r = (16, -1.0)
tome_decr_throughput = tome.utils.benchmark(
    model,
    device=device,
    verbose=True,
    runs=runs,
    batch_size=batch_size,
    input_size=input_size
)
print(f"Throughput improvement: {tome_decr_throughput / baseline_throughput:.2f}x")

## ToMe Visualization
We provide some visualization functions to visualize the effect of ToMe like in our paper.

In [ ]:
# We use the timm augreg models here, but you can use any supported implementation.
model_name = "vit_large_patch16_384"
model = timm.create_model(model_name, pretrained=True)

# Source tracing is necessary for visualization!
tome.patch.timm(model, trace_source=True)

In [ ]:
input_size = model.default_cfg["input_size"][1]

# Make sure the transform is correct for your model!
transform_list = [
    transforms.Resize(int((256 / 224) * input_size), interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(input_size)
]

# The visualization and model need different transforms
transform_vis  = transforms.Compose(transform_list)
transform_norm = transforms.Compose(transform_list + [
    transforms.ToTensor(),
    transforms.Normalize(model.default_cfg["mean"], model.default_cfg["std"]),
])

In [ ]:
img = Image.open("examples/images/husky.png")
img_vis = transform_vis(img)
img_norm = transform_norm(img)

img

## Visualize one image
You can play with the `r` value to get different numbers of tokens at the end of the network.

In [ ]:
model.r = 25
_ = model(img_norm[None, ...])
source = model._tome_info["source"]

print(f"{source.shape[1]} tokens at the end")
tome.make_visualization(img_vis, source, patch_size=16, class_token=True)

## Visualize intermediate results
Setting `r` to be a list can let us control where we stop merging.

In [ ]:
model.r = [25] * 8  # 8 / 24 layers
_ = model(img_norm[None, ...])
source = model._tome_info["source"]

print(f"{source.shape[1]} tokens at the end")
tome.make_visualization(img_vis, source, patch_size=16, class_token=True)

In [ ]:
model.r = [25] * 16  # 16 / 24 layers
_ = model(img_norm[None, ...])
source = model._tome_info["source"]

print(f"{source.shape[1]} tokens at the end")
tome.make_visualization(img_vis, source, patch_size=16, class_token=True)

In [ ]:
model.r = [25] * 22  # 22 / 24 layers
_ = model(img_norm[None, ...])
source = model._tome_info["source"]

print(f"{source.shape[1]} tokens at the end")
tome.make_visualization(img_vis, source, patch_size=16, class_token=True)